# Fase 1: Definición del Proyecto y Entorno Reproducible
## Análisis de Licitaciones Públicas en Chile (Enero 2025)

- **Asignatura:** Programación para la Ciencia de Datos (MCDI-500)
- **Grupo:** Grupo 2
- **Integrantes:**
  - Ernesto Siebert Aguilera
  - Oscar Salgado Sanhueza
  - Bertgil Gonzalez Olmos
  - Daniel Rojas Sánchez
- **Fuente de Datos:** [ChileCompra — Portal de Datos Abiertos](https://datos-abiertos.chilecompra.cl/descargas/ordenes-y-licitaciones) (`data/raw/04LicitacionesEne.csv`)

## 1. Descripción de la Problemática

Las licitaciones públicas constituyen el mecanismo principal mediante el cual las instituciones del Estado de Chile adquieren bienes y contratan servicios a través de la plataforma **Mercado Público (ChileCompra)**. En cada licitación, el organismo registra un **monto estimado** y, al adjudicar, se conocen las **ofertas ganadoras**.

En las licitaciones de ChileCompra de enero de 2025, se desconoce con qué frecuencia el monto estimado registrado coincide con la suma de las ofertas ganadoras. El proyecto mide esa coincidencia y, cuando los montos difieren, cuantifica la brecha por licitación en los casos en que ambos están expresados en la misma moneda, y examina si ambas cambian según el tamaño del proveedor ganador.

## 2. Objetivos del Proyecto

### 2.1. Objetivo General
Analizar la diferencia entre el monto estimado registrado y el total de las ofertas ganadoras en las licitaciones de ChileCompra de enero de 2025, y su relación con el tamaño del proveedor ganador.

### 2.2. Objetivos Específicos
1. Determinar en qué proporción de las licitaciones comparables el monto estimado registrado coincide con el total de las ofertas ganadoras.
2. Describir la magnitud y el sentido de la brecha por licitación cuando ambos montos difieren, separando por moneda.
3. Comparar la coincidencia y la brecha según el tamaño del proveedor ganador.

Seleccionar las licitaciones comparables y calcular la brecha son pasos de la metodología (F2), no objetivos: la brecha es la suma de las ofertas ganadoras menos el monto estimado, y solo se calcula cuando ambos montos son válidos y están en la misma moneda.

## 3. Preguntas de Investigación

1. ¿En qué proporción de las licitaciones comparables coincide el monto estimado registrado con el total de las ofertas ganadoras y, cuando difieren, cuál es la magnitud y el sentido de la brecha en cada moneda?
2. ¿Cambian la coincidencia y la brecha según el tamaño del proveedor ganador (Micro, Pequeña, Mediana o Grande)?

La unidad de análisis es la **licitación**: se filtran las ofertas ganadoras y se suman por licitación. Los valores extremos (IQR) se usan como control descriptivo de los montos, no como pregunta.

## 4. Criterios de Éxito

El proyecto se considerará exitoso si cumple con los siguientes criterios verificables:

1. Generar un conjunto preprocesado sin datos personales (data/processed/04LicitacionesEne_procesado.csv, 45.741 ofertas) y una tabla con una fila por licitación con su brecha (04LicitacionesEne_brecha_por_licitacion.csv, 2.292 licitaciones), validados con aserciones antes de exportar.
2. Documentar el diccionario completo de las 72 variables de entrada en docs/README.md.
3. Comparar montos solo dentro de una misma moneda, sin conversión, y declarar cuántas licitaciones quedan fuera de la comparación y por qué.
4. Garantizar la reproducibilidad: cuadernos ejecutables de principio a fin con el entorno de requirements.txt y reglas en funciones de src/.
5. Responder las dos preguntas con proporciones, distribuciones y tablas por moneda y por tamaño, calculadas en los cuadernos.

## 5. Alcance del Proyecto

- **Cobertura de datos:** El análisis abarca los **45.741 registros de ofertas** (correspondientes a **2.292 licitaciones únicas** y **326 instituciones públicas**) del archivo `04LicitacionesEne.csv` de ChileCompra.
- **Nivel de granularidad:** El archivo se preprocesa a nivel de **oferta por ítem** y el análisis se hace a nivel de **licitación (`NroLicitacion`)**, con la suma de sus ofertas ganadoras.
- **Enfoque analítico:** El proyecto es **descriptivo**: mide la coincidencia y la brecha y las compara por moneda y por tamaño del proveedor ganador. No se entrenan modelos predictivos.

## 6. Restricciones y Limitaciones

1. Datos: un solo mes (enero de 2025); los hallazgos no capturan variaciones estacionales.
2. Monedas: los montos solo se comparan dentro de la misma moneda, sin conversión. CLP concentra el 99,70 % de las ofertas; CLF, USD y UTM se reportan por separado.
3. Unidad de análisis: cada fila del archivo es una oferta por ítem; la brecha se calcula por licitación con la suma de las ofertas ganadoras.
4. Monto neto frente a estimado: las ofertas son montos netos y el estimado puede incluir IVA.
5. Registro del estimado: no se sabe cuándo se registró, así que una coincidencia no prueba que haya sido un presupuesto previo.
6. Tamaño: para comparar por tamaño se excluyen las licitaciones con ganadoras de distintos tamaños o solo NoClasificado, que no tiene rango en la escala.
7. Calidad de origen: hay columnas administrativas sin datos (LicitacionBaseTipo, ContratoRenovable, UnidadTiempoRenovacion) y errores de digitación puntuales (como un monto de -1,0 en CLP).

## 7. Estructura Programática de la Definición del Proyecto

`diccionario PROYECTO` → `presentar_proyecto(PROYECTO)` → `alcance y restricciones`

Se almacena la configuración formal de la Fase 1 en el diccionario `PROYECTO`, se presenta mediante `presentar_proyecto(PROYECTO)` y a continuación se detallan el alcance (`incluye` / `excluye`) y las restricciones declaradas.

In [1]:
# PASO 1: Definir el diccionario de configuración formal del proyecto (PROYECTO).
PROYECTO = {
    "titulo": "Análisis de Licitaciones Públicas en Chile (Enero 2025)",
    "grupo": "Grupo 2",
    "asignatura": "Programación para la Ciencia de Datos (MCDI-500)",
    "integrantes": [
        "Ernesto Siebert Aguilera",
        "Oscar Salgado Sanhueza",
        "Bertgil Gonzalez Olmos",
        "Daniel Rojas Sánchez"
    ],
    "problematica": (
        "En las licitaciones de ChileCompra de enero de 2025, se desconoce con qué frecuencia el "
        "monto estimado registrado coincide con la suma de las ofertas ganadoras. El proyecto "
        "mide esa coincidencia y, cuando los montos difieren, cuantifica la brecha por licitación "
        "en los casos en que ambos están expresados en la misma moneda, y examina si ambas "
        "cambian según el tamaño del proveedor ganador."
    ),
    "objetivo_general": (
        "Analizar la diferencia entre el monto estimado registrado y el total de las ofertas "
        "ganadoras en las licitaciones de ChileCompra de enero de 2025, y su relación con el "
        "tamaño del proveedor ganador."
    ),
    "objetivos_especificos": [
        "Determinar en qué proporción de las licitaciones comparables el monto estimado registrado coincide con el total de las ofertas ganadoras.",
        "Describir la magnitud y el sentido de la brecha por licitación cuando ambos montos difieren, separando por moneda.",
        "Comparar la coincidencia y la brecha según el tamaño del proveedor ganador."
    ],
    "preguntas": [
        "¿En qué proporción de las licitaciones comparables coincide el monto estimado registrado con el total de las ofertas ganadoras y, cuando difieren, cuál es la magnitud y el sentido de la brecha en cada moneda?",
        "¿Cambian la coincidencia y la brecha según el tamaño del proveedor ganador (Micro, Pequeña, Mediana o Grande)?"
    ],
    "criterios_exito": [
        "Generar un conjunto preprocesado sin datos personales (data/processed/04LicitacionesEne_procesado.csv, 45.741 ofertas) y una tabla con una fila por licitación con su brecha (04LicitacionesEne_brecha_por_licitacion.csv, 2.292 licitaciones), validados con aserciones antes de exportar.",
        "Documentar el diccionario completo de las 72 variables de entrada en docs/README.md.",
        "Comparar montos solo dentro de una misma moneda, sin conversión, y declarar cuántas licitaciones quedan fuera de la comparación y por qué.",
        "Garantizar la reproducibilidad: cuadernos ejecutables de principio a fin con el entorno de requirements.txt y reglas en funciones de src/.",
        "Responder las dos preguntas con proporciones, distribuciones y tablas por moneda y por tamaño, calculadas en los cuadernos."
    ],
    "alcance": {
        "incluye": [
            "Definición formal de la problemática, objetivos, preguntas y criterios de éxito",
            "Validación y preprocesamiento de las 45.741 ofertas (2.292 licitaciones) de ChileCompra de enero de 2025",
            "Análisis descriptivo de la coincidencia y la brecha por licitación, por moneda y por tamaño del proveedor ganador"
        ],
        "excluye": [
            "Modelos predictivos o de aprendizaje automático (machine learning)",
            "Conversión entre monedas con tipos de cambio externos para CLF, USD y UTM",
            "Inferencia causal sobre intencionalidad administrativa o colusión"
        ]
    },
    "restricciones": [
        "Datos: un solo mes (enero de 2025); los hallazgos no capturan variaciones estacionales.",
        "Monedas: los montos solo se comparan dentro de la misma moneda, sin conversión. CLP concentra el 99,70 % de las ofertas; CLF, USD y UTM se reportan por separado.",
        "Unidad de análisis: cada fila del archivo es una oferta por ítem; la brecha se calcula por licitación con la suma de las ofertas ganadoras.",
        "Monto neto frente a estimado: las ofertas son montos netos y el estimado puede incluir IVA.",
        "Registro del estimado: no se sabe cuándo se registró, así que una coincidencia no prueba que haya sido un presupuesto previo.",
        "Tamaño: para comparar por tamaño se excluyen las licitaciones con ganadoras de distintos tamaños o solo NoClasificado, que no tiene rango en la escala.",
        "Calidad de origen: hay columnas administrativas sin datos (LicitacionBaseTipo, ContratoRenovable, UnidadTiempoRenovacion) y errores de digitación puntuales (como un monto de -1,0 en CLP)."
    ]
}

In [2]:
# PASO 2: Definir la función para validar e imprimir la definición del proyecto.
def presentar_proyecto(config):
    """
    Imprime la definicion del proyecto de forma legible.

    Parametros
    ----------
    config : dict
        Diccionario de configuracion del proyecto.

    Retorna
    -------
    None

    Lanza
    -----
    KeyError
        Si falta alguna de las claves obligatorias.
    """
    obligatorias = ("titulo", "problematica", "objetivo_general", "objetivos_especificos")
    faltantes = [c for c in obligatorias if c not in config]
    if faltantes:
        # Se falla temprano y con un mensaje que dice QUE falta, no solo que algo falló.
        raise KeyError(f"Faltan claves obligatorias en la configuracion: {faltantes}")

    print(config["titulo"].upper())
    print("=" * 60)
    print(f"{config['asignatura']} · {config['grupo']}")
    print("Integrantes:", ", ".join(config["integrantes"]))
    print("\nProblemática")
    print(config["problematica"])
    print("\nObjetivo general")
    print(config["objetivo_general"])
    print("\nObjetivos específicos")
    for i, obj in enumerate(config["objetivos_especificos"], start=1):
        print(f"  {i}. {obj}")
    print("\nPreguntas que orientan el trabajo")
    for pregunta in config["preguntas"]:
        print(f"  · {pregunta}")
    print("\nCriterios de éxito")
    for i, criterio in enumerate(config["criterios_exito"], start=1):
        print(f"  {i}. {criterio}")


presentar_proyecto(PROYECTO)

ANÁLISIS DE LICITACIONES PÚBLICAS EN CHILE (ENERO 2025)
Programación para la Ciencia de Datos (MCDI-500) · Grupo 2
Integrantes: Ernesto Siebert Aguilera, Oscar Salgado Sanhueza, Bertgil Gonzalez Olmos, Daniel Rojas Sánchez

Problemática
En las licitaciones de ChileCompra de enero de 2025, se desconoce con qué frecuencia el monto estimado registrado coincide con la suma de las ofertas ganadoras. El proyecto mide esa coincidencia y, cuando los montos difieren, cuantifica la brecha por licitación en los casos en que ambos están expresados en la misma moneda, y examina si ambas cambian según el tamaño del proveedor ganador.

Objetivo general
Analizar la diferencia entre el monto estimado registrado y el total de las ofertas ganadoras en las licitaciones de ChileCompra de enero de 2025, y su relación con el tamaño del proveedor ganador.

Objetivos específicos
  1. Determinar en qué proporción de las licitaciones comparables el monto estimado registrado coincide con el total de las ofertas g

In [3]:
# PASO 3: Imprimir el alcance (incluye / excluye) y las restricciones declaradas.
print("Alcance del proyecto")
print("  Incluye:", ", ".join(PROYECTO["alcance"]["incluye"]))
print("  Excluye:", ", ".join(PROYECTO["alcance"]["excluye"]))

print("\nRestricciones y limitaciones declaradas")
for restriccion in PROYECTO["restricciones"]:
    print(f"  · {restriccion}")

Alcance del proyecto
  Incluye: Definición formal de la problemática, objetivos, preguntas y criterios de éxito, Validación y preprocesamiento de las 45.741 ofertas (2.292 licitaciones) de ChileCompra de enero de 2025, Análisis descriptivo de la coincidencia y la brecha por licitación, por moneda y por tamaño del proveedor ganador
  Excluye: Modelos predictivos o de aprendizaje automático (machine learning), Conversión entre monedas con tipos de cambio externos para CLF, USD y UTM, Inferencia causal sobre intencionalidad administrativa o colusión

Restricciones y limitaciones declaradas
  · Datos: un solo mes (enero de 2025); los hallazgos no capturan variaciones estacionales.
  · Monedas: los montos solo se comparan dentro de la misma moneda, sin conversión. CLP concentra el 99,70 % de las ofertas; CLF, USD y UTM se reportan por separado.
  · Unidad de análisis: cada fila del archivo es una oferta por ítem; la brecha se calcula por licitación con la suma de las ofertas ganadoras.
  · 

## 8. Verificación del Entorno

Antes de trabajar con datos se comprueba que el cuaderno se ejecuta con el entorno virtual del proyecto (`.venv`, kernel **Python (grupo2-mcdi500)**) y que las librerías declaradas en `requirements.txt` están disponibles. La configuración del entorno se describe en la sección *Configurar el entorno* del `README.md`; aquí solo se verifica.

In [4]:
# PASO 4: Definir y ejecutar la verificación del entorno.
# Solo usa la biblioteca estándar: puede copiarse como primera celda de cualquier cuaderno.
import importlib
import platform
import sys
from pathlib import Path

# Librerías que importan src/ y los notebooks (las mismas de requirements.txt).
LIBRERIAS_PROYECTO = ["numpy", "pandas"]


def verificar_entorno(librerias):
    '''
    Muestra y devuelve el estado del entorno de ejecución del cuaderno.

    Parámetros
    ----------
    librerias : list of str
        Nombres de importación de las librerías que el cuaderno necesita.

    Retorna
    -------
    dict
        Intérprete, si hay entorno virtual activo, carpeta de trabajo, versión de
        Python, sistema operativo y versión de cada librería (None si falta).
    '''
    # El intérprete que ejecuta ESTE cuaderno. Si no es el de .venv,
    # el kernel es el equivocado: Kernel -> Change Kernel.
    en_venv = sys.prefix != sys.base_prefix
    reporte = {
        "interprete": sys.executable,
        "entorno_virtual": en_venv,
        # Las rutas relativas se resuelven desde aquí, no desde la raíz del proyecto.
        "carpeta_trabajo": str(Path.cwd()),
        "python": platform.python_version(),
        "sistema": f"{platform.system()} {platform.release()}",
        "librerias": {},
    }

    print("Intérprete        :", reporte["interprete"])
    print("Entorno virtual   :", "[OK] activo" if en_venv else "[AVISO] se usa el Python del sistema")
    print("Carpeta de trabajo:", reporte["carpeta_trabajo"])
    print("Python            :", reporte["python"])
    print("Sistema operativo :", reporte["sistema"])

    print("\nLibrerías")
    for nombre in librerias:
        try:
            version = importlib.import_module(nombre).__version__
            print(f"  [OK]    {nombre:12} {version}")
        except ImportError:
            version = None
            print(f"  [FALTA] {nombre:12} instalar con: python -m pip install -r requirements.txt")
        reporte["librerias"][nombre] = version

    return reporte


ENTORNO = verificar_entorno(LIBRERIAS_PROYECTO)

Intérprete        : /Users/oscar/Documents/Magister CS e IA/repositorio/proyecto-grupo2-mcdi500/.venv/bin/python
Entorno virtual   : [OK] activo
Carpeta de trabajo: /Users/oscar/Documents/Magister CS e IA/repositorio/proyecto-grupo2-mcdi500/F1
Python            : 3.12.3
Sistema operativo : Darwin 25.5.0

Librerías
  [OK]    numpy        2.5.4


  [OK]    pandas       3.1.0


**Lectura del resultado.** La carpeta de trabajo aparece como `.../F1` porque Jupyter abre el kernel en la carpeta del cuaderno; es el comportamiento esperado. Si el entorno virtual aparece con `[AVISO]` o alguna librería con `[FALTA]`, el cuaderno se está ejecutando con otro intérprete: se debe cambiar el kernel a **Python (grupo2-mcdi500)** o repetir los pasos de *Configurar el entorno* del `README.md`.

La celda siguiente convierte esa lectura en una comprobación: si el entorno no es el correcto, el cuaderno se detiene antes de continuar.

In [5]:
# PASO 5: Detener el cuaderno si el entorno no es el del proyecto.
faltantes = [nombre for nombre, version in ENTORNO["librerias"].items() if version is None]

assert ENTORNO["entorno_virtual"], (
    "El cuaderno no se ejecuta en un entorno virtual. "
    "Cambie el kernel a 'Python (grupo2-mcdi500)'."
)
assert not faltantes, f"Faltan librerías en el entorno: {faltantes}"

print("[OK] Entorno virtual activo")
print(f"[OK] {len(ENTORNO['librerias'])} librerías disponibles")

[OK] Entorno virtual activo
[OK] 2 librerías disponibles


## 9. Reproducibilidad: Dependencias y Exclusiones

Se localiza la raíz del proyecto y se comprueba que `requirements.txt` refleje exactamente el entorno instalado y que `.gitignore` excluya lo que no debe versionarse. La existencia de las carpetas y archivos del proyecto se comprueba en la validación final (sección 12).

Estos archivos ya existen y se mantienen a mano: el cuaderno **solo los lee**, no los genera ni los modifica.

### 9.1. Dependencias: `requirements.txt`

`requirements.txt` declara solo las dependencias directas del proyecto, con versión fija: `numpy` y `pandas`, que importan `src/` y los notebooks, e `ipykernel` y `jupyterlab`, que permiten ejecutar los notebooks. `pip` instala las dependencias indirectas de cada una. El procedimiento para actualizarlo está en el `README.md`.

La comprobación compara cada paquete declarado con la versión instalada en el entorno activo. Si alguien instaló o actualizó una librería sin regenerar el archivo, aparece aquí como diferencia.

In [6]:
# PASO 6: Configurar las rutas del proyecto y comparar requirements.txt con el entorno.
from importlib import metadata
import pandas as pd

# Jupyter abre el kernel en F1/: si ahí no está data/, la raíz es la carpeta superior.
RAIZ = Path.cwd().resolve()
if not (RAIZ / "data").is_dir():
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
DIR_CRUDO = RAIZ / "data" / "raw"
DIR_PROCESADO = RAIZ / "data" / "processed"


def normalizar_paquete(nombre):
    '''Normaliza un nombre de paquete (PEP 503): minúsculas y separadores como "-".'''
    return nombre.strip().lower().replace("_", "-").replace(".", "-")


def leer_requirements(ruta):
    '''Devuelve {paquete: versión} a partir de las líneas `paquete==versión`.'''
    declarados = {}
    for linea in Path(ruta).read_text(encoding="utf-8").splitlines():
        linea = linea.strip()
        if linea and not linea.startswith("#") and "==" in linea:
            nombre, version = linea.split("==", 1)
            declarados[normalizar_paquete(nombre)] = version.strip()
    return declarados


def comparar_requirements(declarados):
    '''
    Compara las versiones declaradas con las instaladas.

    Retorna
    -------
    DataFrame
        Paquetes cuya versión no coincide o que no están instalados (vacío si todo coincide).
    '''
    instalados = {normalizar_paquete(d.metadata["Name"]): d.version for d in metadata.distributions()}
    diferencias = [
        {"paquete": p, "declarada": v, "instalada": instalados.get(p)}
        for p, v in sorted(declarados.items())
        if instalados.get(p) != v
    ]
    return pd.DataFrame(diferencias, columns=["paquete", "declarada", "instalada"])


REQUIREMENTS = leer_requirements(RAIZ / "requirements.txt")
DIFERENCIAS_REQ = comparar_requirements(REQUIREMENTS)

print(f"Paquetes declarados en requirements.txt: {len(REQUIREMENTS)}")
print(f"Paquetes con versión distinta o ausentes: {len(DIFERENCIAS_REQ)}")
if not DIFERENCIAS_REQ.empty:
    display(DIFERENCIAS_REQ)

# Librerías del proyecto (PASO 4) frente a lo declarado.
display(pd.DataFrame([
    {"libreria": lib,
     "declarada": REQUIREMENTS.get(lib),
     "instalada": ENTORNO["librerias"][lib]}
    for lib in LIBRERIAS_PROYECTO
]))

Paquetes declarados en requirements.txt: 4
Paquetes con versión distinta o ausentes: 0


,libreria,declarada,instalada
0,numpy,2.5.4,2.5.4
1,pandas,3.1.0,3.1.0


### 9.2. Exclusiones: `.gitignore`

El entorno virtual y las cachés se regeneran en cada equipo y no deben subirse al repositorio; las variables de entorno (`.env`) pueden contener credenciales. El archivo de entrada `data/raw/04LicitacionesEne.csv` tampoco se versiona, porque contiene RUT y nombres de proveedores: se comprueba que esté excluido y se informa solo como archivo local (su descarga se documenta en `data/raw/README.md`). Los archivos de `data/processed/` se revisan contra el límite de 100 MB de GitHub.

In [7]:
# PASO 7: Revisar las exclusiones de .gitignore y el tamaño de los archivos de datos.
LIMITE_GITHUB_MB = 100
EXCLUSIONES_ESPERADAS = [
    ".venv", "__pycache__", ".ipynb_checkpoints", ".env",
    "data/raw/*.csv", "data/raw/*.zip",  # datos de entrada con RUT: no se versionan
]

reglas_gitignore = {
    linea.strip().rstrip("/")
    for linea in (RAIZ / ".gitignore").read_text(encoding="utf-8").splitlines()
    if linea.strip() and not linea.strip().startswith("#")
}

print(".gitignore")
for patron in EXCLUSIONES_ESPERADAS:
    print(f"  {'[OK]   ' if patron in reglas_gitignore else '[FALTA]'} excluye {patron}")

print("\nArchivos CSV en data/processed/ (límite de GitHub por archivo: 100 MB)")
TAMANOS_DATOS = {}
for archivo in sorted(DIR_PROCESADO.glob("*.csv")):
    tamano = archivo.stat().st_size / 1024**2
    TAMANOS_DATOS[archivo.relative_to(RAIZ).as_posix()] = tamano
    estado = "[OK]   " if tamano < LIMITE_GITHUB_MB else "[FALTA]"
    print(f"  {estado} {archivo.relative_to(RAIZ).as_posix():50} {tamano:6.1f} MB")

print("\nDatos de entrada en data/raw/ (locales, no versionados; ver data/raw/README.md)")
for archivo in sorted(DIR_CRUDO.glob("*.csv")):
    tamano = archivo.stat().st_size / 1024**2
    print(f"  [LOCAL] {archivo.relative_to(RAIZ).as_posix():50} {tamano:6.1f} MB")

.gitignore
  [OK]    excluye .venv
  [OK]    excluye __pycache__
  [OK]    excluye .ipynb_checkpoints
  [OK]    excluye .env
  [OK]    excluye data/raw/*.csv
  [OK]    excluye data/raw/*.zip

Archivos CSV en data/processed/ (límite de GitHub por archivo: 100 MB)
  [OK]    data/processed/04LicitacionesEne_brecha_por_licitacion.csv    0.3 MB
  [OK]    data/processed/04LicitacionesEne_procesado.csv        7.0 MB
  [OK]    data/processed/diccionario_f3.csv                     0.0 MB
  [OK]    data/processed/parametros_f3.csv                      0.0 MB

Datos de entrada en data/raw/ (locales, no versionados; ver data/raw/README.md)
  [LOCAL] data/raw/04LicitacionesEne.csv                       56.8 MB


## 10. Ficha Técnica del Conjunto de Datos y Diccionario de Variables

A continuación se declara programáticamente la **ficha técnica (`FICHA`)** del archivo de entrada `data/raw/04LicitacionesEne.csv` —incluyendo el conteo de variables por **rol analítico** (`continua`, `discreta`, `binaria`, `nominal`, `ordinal`, `fecha`, `alta_cardinalidad`, `identificador`)— y el **diccionario de las 72 variables (`diccionario`)**.

> **Nota metodológica sobre el rol analítico vs. el validador automático (`src/validar_dataset.py`):** El diccionario y `FICHA["roles"]` declaran el **rol analítico conceptual** de cada variable (el que determina su preprocesamiento en la Fase 2), el cual no coincide necesariamente con el tipo físico que trae el CSV crudo. En particular:
> - `MontoNetoOferta` y `MontoTotalOferta` tienen rol `continua` (aunque en el CSV original vienen como texto con coma decimal y el validador heurístico los detecta inicialmente como `alta cardinalidad`).
> - `TamanoProveedor` tiene rol `ordinal` (`Micro` < `Pequeña` < `Mediana` < `Grande`), mientras que en crudo se lee como texto nominal. `NoClasificado` no es un tamaño: queda fuera de la escala, como faltante.
> - Las 3 columnas con 100% de nulos en este corte (`LicitacionBaseTipo`, `ContratoRenovable` y `UnidadTiempoRenovacion`) se registran en el diccionario bajo su rol administrativo esperado (`nominal`), mientras que `validar_dataset.py` las reporta en crudo como `vacía`.

In [8]:
# PASO 8: Importar utilidades y declarar la FICHA técnica.
# RAIZ y sys.path se configuran en el PASO 6.
from src import utilidades

ARCHIVO = RAIZ / "data" / "raw" / "04LicitacionesEne.csv"

FICHA = {
    "titulo": "Licitaciones Públicas de ChileCompra — Enero 2025",
    "autor": "Dirección ChileCompra",
    "plataforma": "Portal de Datos Abiertos de ChileCompra",
    "url": "https://datos-abiertos.chilecompra.cl/descargas/ordenes-y-licitaciones",
    "licencia_abierta": True,
    "unidad_observacion": "Oferta de un proveedor para un ítem dentro de una licitación pública",
    "filas": 45741,
    "columnas": 72,
    "tamano_mb": 56.8,
    "descarga": "data/raw/README.md (archivo local, no versionado)",
    # Conteo de variables por ROL ANALÍTICO, no por tipo de dato: es el rol
    # el que determina qué preprocesamiento habrá que justificar en la Fase 2.
    "roles": {
        "continua": 3,           # MontoEstimadoLicitacion, MontoNetoOferta, MontoTotalOferta
        "discreta": 4,           # TiempoDuracionContrato, ValorTiempoRenovacion, CantidadItem, CantidadOferta
        "binaria": 12,           # MontoEstimadoVisible, ContemplaObrasPublicas, TipoConvocatoria, ResultadoOferta, etc.
        "nominal": 20,           # TipoLicitacion, MonedaLicitacion, EstadoLicitacion, MonedaOferta, RubroN1, etc.
        "ordinal": 1,            # TamanoProveedor (Micro < Pequeña < Mediana < Grande; NoClasificado fuera de la escala)
        "fecha": 12,             # FechaPublicacion, FechaCierre, FechaAdjudicacion, etc.
        "alta_cardinalidad": 15, # Institucion, ActividadProveedor, RubroN2/N3, textos descriptivos, etc.
        "identificador": 5,      # NroLicitacion, NumeroActaAprobacion, UnidadCompraRUT, entCode, CodigoProductoONU
    },
    "pct_nulos_max_variable": 100.0,  # LicitacionBaseTipo, ContratoRenovable, UnidadTiempoRenovacion
    "pct_nulos_min_no_cero": 0.2,     # UnidadTiempoEvaluacion (0.19%), FechaAdjudicacion (0.20%), DescripcionItem (0.20%)
}

for clave, valor in FICHA.items():
    if clave == "roles":
        print("roles:")
        for rol, cant in valor.items():
            print(f"  · {rol:18s}: {cant}")
    else:
        print(f"{clave:22s}: {valor}")

titulo                : Licitaciones Públicas de ChileCompra — Enero 2025
autor                 : Dirección ChileCompra
plataforma            : Portal de Datos Abiertos de ChileCompra
url                   : https://datos-abiertos.chilecompra.cl/descargas/ordenes-y-licitaciones
licencia_abierta      : True
unidad_observacion    : Oferta de un proveedor para un ítem dentro de una licitación pública
filas                 : 45741
columnas              : 72
tamano_mb             : 56.8
descarga              : data/raw/README.md (archivo local, no versionado)
roles:
  · continua          : 3
  · discreta          : 4
  · binaria           : 12
  · nominal           : 20
  · ordinal           : 1
  · fecha             : 12
  · alta_cardinalidad : 15
  · identificador     : 5
pct_nulos_max_variable: 100.0
pct_nulos_min_no_cero : 0.2


In [9]:
# PASO 9: Construir el diccionario de las 72 variables con su tipo y rol analítico.
VARIABLES_FASE1 = [
    # 1 a 10: Identificación, clasificación y montos estimados de la licitación
    ("NroLicitacion", "str", "identificador", "Código único de la licitación en Mercado Público"),
    ("NombreLicitacion", "str", "alta_cardinalidad", "Título o nombre asignado al proceso licitatorio"),
    ("TipoLicitacion", "str", "nominal", "Modalidad y tramo en UTM de la licitación (L1, LE, LP, LQ, LR)"),
    ("Descripcion", "str", "alta_cardinalidad", "Descripción general del requerimiento de la licitación"),
    ("MonedaLicitacion", "str", "nominal", "Moneda del presupuesto de la licitación (CLP, CLF, USD, UTM)"),
    ("MontoEstimadoLicitacion", "int64", "continua", "Presupuesto o monto total estimado para la licitación"),
    ("MontoEstimadoVisible", "str", "binaria", "Indica si el monto estimado fue visible para los oferentes (Si/No)"),
    ("BaseEstimacionMontoLicitacion", "str", "nominal", "Criterio empleado para estimar el monto de la licitación"),
    ("FuenteFinanciamiento", "str", "nominal", "Origen de los fondos con que se financia la compra"),
    ("JustificacionMontoEstimado", "str", "alta_cardinalidad", "Glosa o código de justificación del presupuesto estimado"),
    # 11 a 21: Cronograma e hitos temporales
    ("FechaPublicacion", "datetime64[ns]", "fecha", "Fecha y hora de publicación del llamado a licitación"),
    ("FechaInicioPreguntas", "str", "fecha", "Fecha y hora de apertura del foro de consultas"),
    ("FechaFinalPreguntas", "str", "fecha", "Fecha y hora de cierre para la recepción de preguntas"),
    ("FechaPublicacionRespuestas", "str", "fecha", "Fecha y hora comprometida para publicar respuestas"),
    ("FechaActoAperturaTecnica", "str", "fecha", "Fecha y hora de apertura de las ofertas técnicas"),
    ("FechaActoAperturaEconomica", "str", "fecha", "Fecha y hora de apertura de las ofertas económicas"),
    ("FechaCierre", "datetime64[ns]", "fecha", "Fecha y hora límite para presentar ofertas"),
    ("FechaAdjudicacion", "datetime64[ns]", "fecha", "Fecha y hora efectiva de adjudicación de la licitación"),
    ("FechaEntregaEnSoporteFisico", "str", "fecha", "Fecha límite de entrega de antecedentes físicos"),
    ("FechaEstimadaEvaluacionOfertas", "str", "fecha", "Fecha estimada de evaluación de las ofertas"),
    ("UnidadTiempoEvaluacion", "str", "nominal", "Unidad de medida del plazo de evaluación"),
    # 22 a 48: Condiciones administrativas, adjudicación y contrato
    ("EstadoLicitacion", "str", "nominal", "Estado del proceso (Adjudicada, Cerrada, Desierta, Suspendida)"),
    ("ContemplaObrasPublicas", "str", "binaria", "Indica si contempla ejecución de obras públicas (Si/No)"),
    ("LicitacionInformada", "str", "nominal", "Indicador de licitación informada en el sistema"),
    ("LicitacionBaseTipo", "float64", "nominal", "Indicador de uso de Bases Tipo de Contraloría"),
    ("TipoAdjudicacion", "str", "nominal", "Modalidad administrativa de adjudicación"),
    ("TipoAprobacionAdjudicacion", "str", "nominal", "Tipo de acto administrativo que aprueba la adjudicación"),
    ("NumeroActaAprobacion", "str", "identificador", "Número o folio del acto que aprueba la adjudicación"),
    ("FechaActaAprobacion", "str", "fecha", "Fecha del acto o resolución de aprobación"),
    ("TipoConvocatoria", "str", "binaria", "Carácter de la convocatoria (Abierta o Cerrada)"),
    ("NroEtapasLicitacion", "str", "binaria", "Cantidad de etapas de apertura (1 o 2 etapas)"),
    ("SubContratacion", "str", "binaria", "Indica si las bases permiten subcontratación (Si/No)"),
    ("ProhibicionSubContratacion", "str", "alta_cardinalidad", "Motivo o detalle de restricciones a la subcontratación"),
    ("TomaRazonContraloria", "str", "binaria", "Indica si requiere Toma de Razón por Contraloría (Si/No)"),
    ("PublicidadOfertasTecnicas", "str", "binaria", "Indica si las ofertas técnicas son públicas tras el cierre"),
    ("RazonPublicidadOfertasTecnicas", "str", "nominal", "Justificación en caso de reserva de ofertas técnicas"),
    ("Contrato", "str", "nominal", "Indica si requiere formalización de contrato u Orden de Compra"),
    ("TiempoDuracionContrato", "int64", "discreta", "Valor entero de la duración del contrato"),
    ("UnidadTiempoDuracionContrato", "str", "nominal", "Unidad de tiempo de la duración del contrato"),
    ("ContratoRenovable", "float64", "nominal", "Indica si el contrato es renovable"),
    ("ValorTiempoRenovacion", "int64", "discreta", "Plazo entero de renovación del contrato"),
    ("UnidadTiempoRenovacion", "float64", "nominal", "Unidad de tiempo de renovación del contrato"),
    ("FechaEstimadaFirmaContrato", "str", "fecha", "Fecha estimada para la suscripción del contrato"),
    ("TipoEjecucion", "str", "binaria", "Modalidad de ejecución del contrato"),
    ("PlazoPagoContrato", "str", "binaria", "Plazo establecido para el pago de facturas"),
    ("TipoPago", "str", "nominal", "Modalidad de pago definida en las bases"),
    ("ObservacionContrato", "str", "alta_cardinalidad", "Observaciones adicionales sobre condiciones contractuales"),
    ("ExtensionPlazo", "str", "binaria", "Indica si la licitación tuvo extensión del plazo de cierre (Si/No)"),
    # 49 a 53: Organismo comprador
    ("UnidadCompra", "str", "alta_cardinalidad", "Nombre de la unidad de compra dentro de la institución"),
    ("UnidadCompraRUT", "str", "identificador", "RUT de la entidad o unidad compradora"),
    ("entCode", "int64", "identificador", "Código numérico único de la institución compradora"),
    ("Institucion", "str", "alta_cardinalidad", "Nombre oficial de la institución u organismo público comprador"),
    ("Sector", "str", "nominal", "Sector del Estado al que pertenece el organismo comprador"),
    # 54 a 62: Rubro e ítem licitado (Clasificador ONU)
    ("RubroN1", "str", "nominal", "Categoría de rubro de nivel 1 según clasificador UNSPSC/ONU"),
    ("RubroN2", "str", "alta_cardinalidad", "Categoría de rubro de nivel 2 (familia de productos/servicios)"),
    ("RubroN3", "str", "alta_cardinalidad", "Categoría de rubro de nivel 3 (clase de productos/servicios)"),
    ("CodigoProductoONU", "int64", "identificador", "Código numérico UNSPSC del producto o servicio licitado"),
    ("ONUProducto", "str", "alta_cardinalidad", "Glosa estándar del producto según clasificador ONU"),
    ("NombreItem", "str", "alta_cardinalidad", "Nombre del ítem o línea de producto solicitado"),
    ("DescripcionItem", "str", "alta_cardinalidad", "Especificación técnica detallada solicitada para el ítem"),
    ("UnidadMedida", "str", "nominal", "Unidad de medida del ítem requerido"),
    ("CantidadItem", "int64", "discreta", "Cantidad entera de unidades solicitadas por el comprador"),
    # 63 a 72: Proveedor y detalle de la oferta presentada
    ("ActividadProveedor", "str", "alta_cardinalidad", "Giro o actividad económica declarada por el proveedor"),
    ("TamanoProveedor", "category", "ordinal", "Tamaño empresarial ordenado (Micro < Pequeña < Mediana < Grande); NoClasificado queda fuera de la escala"),
    ("NombreOferta", "str", "alta_cardinalidad", "Título dado por el proveedor a su propuesta comercial"),
    ("EspecificacionesProveedor", "str", "alta_cardinalidad", "Detalle técnico de lo ofertado por el proveedor"),
    ("EstadoOferta", "str", "binaria", "Admisibilidad de la oferta en apertura (Aceptada o Rechazada)"),
    ("CantidadOferta", "int64", "discreta", "Cantidad entera de unidades ofertadas por el proveedor"),
    ("MonedaOferta", "str", "nominal", "Moneda en que se expresa la oferta (CLP, CLF, USD, UTM)"),
    ("MontoNetoOferta", "Float64", "continua", "Precio unitario neto ofertado por el proveedor"),
    ("MontoTotalOferta", "Float64", "continua", "Monto total neto de la línea ofertada"),
    ("ResultadoOferta", "str", "binaria", "Resultado final de la línea ofertada (Ganadora o Perdedora)"),
]

diccionario = pd.DataFrame(
    VARIABLES_FASE1,
    columns=["variable", "tipo_esperado", "rol_analitico", "descripcion"]
)

conteo_roles = diccionario["rol_analitico"].value_counts().to_dict()
assert conteo_roles == FICHA["roles"], f"Desajuste en roles: {conteo_roles} vs {FICHA['roles']}"

print(f"Total de variables documentadas: {len(diccionario)}")
print("\nDistribución por rol analítico (coherente con FICHA['roles']):")
print(diccionario["rol_analitico"].value_counts().to_string())
diccionario.head(10)

Total de variables documentadas: 72

Distribución por rol analítico (coherente con FICHA['roles']):
rol_analitico
nominal              20
alta_cardinalidad    15
binaria              12
fecha                12
identificador         5
discreta              4
continua              3
ordinal               1


,variable,tipo_esperado,rol_analitico,descripcion
0,NroLicitacion,str,identificador,Código único de la licitación en Mercado Público
1,NombreLicitacion,str,alta_cardinalidad,Título o nombre asignado al proceso licitatorio
2,TipoLicitacion,str,nominal,"Modalidad y tramo en UTM de la licitación (L1,..."
3,Descripcion,str,alta_cardinalidad,Descripción general del requerimiento de la li...
4,MonedaLicitacion,str,nominal,"Moneda del presupuesto de la licitación (CLP, ..."
5,MontoEstimadoLicitacion,int64,continua,Presupuesto o monto total estimado para la lic...
6,MontoEstimadoVisible,str,binaria,Indica si el monto estimado fue visible para l...
7,BaseEstimacionMontoLicitacion,str,nominal,Criterio empleado para estimar el monto de la ...
8,FuenteFinanciamiento,str,nominal,Origen de los fondos con que se financia la co...
9,JustificacionMontoEstimado,str,alta_cardinalidad,Glosa o código de justificación del presupuest...


## 11. Pruebas de los Módulos Compartidos (`src/utilidades.py` y `src/preprocesamiento.py`)

Se comprueba el comportamiento de las funciones de normalización en tres escenarios: **caso normal**, **caso límite** (signos consecutivos y lista vacía) y **manejo de excepciones** (`ProyectoError` ante tipos inválidos o colisión de nombres duplicados).

También se prueba la escala ordinal del tamaño de proveedor, que define `src/preprocesamiento.py` y usan F2 y F3: `Micro` < `Pequeña` < `Mediana` < `Grande`. `NoClasificado` queda como faltante porque no tiene rango, y una categoría desconocida detiene la ejecución con `ValueError` en lugar de perderse sin aviso.

In [10]:
# PASO 10: Ejecutar pruebas del módulo src/utilidades.py (caso normal, límite y excepciones).
# --- Caso normal ---
assert utilidades.normalizar_nombre("Avg Glucose Level") == "avg_glucose_level"
assert utilidades.normalizar_columnas(["A B", "c-d"]) == ["a_b", "c_d"]
print("[OK] Caso normal: normalización correcta")

# --- Caso límite: signos consecutivos y espacios en los extremos ---
assert utilidades.normalizar_nombre("  ¿BMI (kg/m2)?  ") == "bmi_kg_m2"
assert utilidades.normalizar_columnas([]) == []
print("[OK] Caso límite: signos y lista vacía")

# --- Excepción 1: tipo incorrecto ---
try:
    utilidades.normalizar_nombre(42)
except utilidades.ProyectoError as e:
    print(f"[OK] Excepción capturada (tipo): {e}")

# --- Excepción 2: la normalización colapsa dos nombres en uno ---
# "BMI" y " bmi " producen ambos "bmi": el módulo lo detecta en vez de perder una columna.
try:
    utilidades.normalizar_columnas(["BMI", " bmi "])
except utilidades.ProyectoError as e:
    print(f"[OK] Excepción capturada (duplicados): {e}")

# --- Escala ordinal del tamaño de proveedor (src/preprocesamiento.py) ---
from src import preprocesamiento

tamanos = preprocesamiento.ordenar_tamano(
    pd.DataFrame({"TamanoProveedor": ["Grande", "Micro", "NoClasificado"]})
)["TamanoProveedorOrdinal"]
assert list(tamanos.cat.categories) == ["Micro", "Pequeña", "Mediana", "Grande"]
rangos = tamanos.cat.codes  # posición en la escala; -1 = sin rango
assert rangos.iloc[1] < rangos.iloc[0]
assert rangos.iloc[2] == -1 and pd.isna(tamanos.iloc[2])
print("[OK] Escala ordinal: Micro < Grande y NoClasificado fuera de la escala")

try:
    preprocesamiento.ordenar_tamano(pd.DataFrame({"TamanoProveedor": ["Gigante"]}))
except ValueError as e:
    print(f"[OK] Excepción capturada (categoría desconocida): {e}")
else:
    raise AssertionError("Una categoría desconocida debía detener la ejecución.")


[OK] Caso normal: normalización correcta
[OK] Caso límite: signos y lista vacía
[OK] Excepción capturada (tipo): Se esperaba un nombre de tipo 'str', pero se recibió 'int'.
[OK] Excepción capturada (duplicados): La normalización colapsa múltiples columnas en el mismo nombre: ['bmi']
[OK] Escala ordinal: Micro < Grande y NoClasificado fuera de la escala
[OK] Excepción capturada (categoría desconocida): Categorías de TamanoProveedor sin regla: ['Gigante']


## 12. Validación Automática de la Fase 1

Finalmente, se ejecuta la función `validar_fase1(RAIZ, PROYECTO, FICHA, diccionario)` para comprobar que la estructura de carpetas, los artefactos del entorno reproducible, la definición del problema, el diccionario de variables y el módulo `src/utilidades.py` cumplen con todos los requisitos de la Fase 1.

In [11]:
# PASO 11: Validar que la Fase 1 cumple todos los requisitos exigidos.
def validar_fase1(raiz, config, ficha, diccionario):
    """
    Comprueba que la Fase 1 dejo lo que la rubrica exige.
    Lanza AssertionError si alguna comprobacion falla.
    """
    print("VALIDACION DE LA FASE 1")
    print("-" * 45)

    # 1. Estructura de carpetas
    esperadas = ["data/raw", "data/processed", "docs", "src", "F1", "F2", "F3", "F4"]
    faltan = [c for c in esperadas if not (raiz / c).is_dir()]
    assert not faltan, f"Faltan carpetas: {faltan}"
    print(f"[OK] Estructura completa ({len(esperadas)} carpetas)")

    # 2. Artefactos del entorno reproducible
    for nombre in (".gitignore", "requirements.txt"):
        archivo = raiz / nombre
        assert archivo.exists(), f"Falta {nombre}."
        assert archivo.stat().st_size > 0, f"{nombre} está vacío."
        print(f"[OK] {nombre} presente ({archivo.stat().st_size} bytes)")

    # 3. Definición del problema
    for clave in ("titulo", "problematica", "objetivo_general", "objetivos_especificos"):
        assert config.get(clave), f"La configuración no declara '{clave}'."
    assert len(config["objetivos_especificos"]) >= 3, "Se esperan al menos tres objetivos."
    print("[OK] Definición del problema completa")

    # 4. Coherencia entre ficha y diccionario
    assert len(diccionario) == ficha["columnas"], "Diccionario y ficha no coinciden."
    roles_declarados = set(diccionario["rol_analitico"])
    assert roles_declarados, "El diccionario no declara roles analíticos."
    print(f"[OK] Diccionario coherente ({len(diccionario)} variables, "
          f"{len(roles_declarados)} roles)")

    # 5. Módulo importable
    assert (raiz / "src" / "utilidades.py").exists(), "Falta el módulo src/utilidades.py."
    assert callable(getattr(utilidades, "normalizar_columnas", None)), \
        "El módulo no expone normalizar_columnas."
    print("[OK] Módulo src/utilidades.py importable")

    # 6. Archivo de entrada (local, no versionado): mismas columnas que el diccionario
    if ARCHIVO.exists():
        encabezado = list(pd.read_csv(ARCHIVO, sep=";", encoding="latin-1", nrows=0).columns)
        assert encabezado == list(diccionario["variable"]), (
            "El encabezado de data/raw/04LicitacionesEne.csv no coincide con el diccionario: "
            "descarga la versión indicada en data/raw/README.md."
        )
        print(f"[OK] Archivo de entrada coincide con el diccionario ({len(encabezado)} columnas)")
    else:
        print("[AVISO] Archivo de entrada no descargado: ver data/raw/README.md")

    print(f"\n[INFO] Conjunto declarado: {ficha['titulo']}")
    return True


validar_fase1(RAIZ, PROYECTO, FICHA, diccionario)

VALIDACION DE LA FASE 1
---------------------------------------------
[OK] Estructura completa (8 carpetas)
[OK] .gitignore presente (5085 bytes)
[OK] requirements.txt presente (249 bytes)
[OK] Definición del problema completa
[OK] Diccionario coherente (72 variables, 8 roles)
[OK] Módulo src/utilidades.py importable
[OK] Archivo de entrada coincide con el diccionario (72 columnas)

[INFO] Conjunto declarado: Licitaciones Públicas de ChileCompra — Enero 2025


True